# FireAID Cohort 2: Post-Fire Burn Severity & Recovery Modeling
## Prototype Pipeline: Shovel Creek Fire (2019), Interior Alaska

- **Student:** Muhammad Saffiullah (UAA / FireAID)
- **Mentor:** Dr. Santosh Panda (UAF)
- **GEE Cloud Project:** `epscor-ioc`

### Workflow Overview:
1. **Fire Boundary Ingestion:** Ingest perimeter for the landmark *Shovel Creek Fire (2019)* (~22,487 acres) from accessible Alaska Fire History.
2. **Harmonized Satellite Compositing:** Pre-fire (2018) & extended post-fire (2020) peak summer composites using NASA Harmonized Landsat-Sentinel (HLS) 30m with Fmask cloud/shadow screening.
3. **Burn Severity Calculation:** Compute dNBR and RdNBR, classify into initial dNBR severity bins requiring local validation, and visualize with interactive `geemap`.
4. **Environmental Covariates:** Extract Google Dynamic World 10m pre-fire landcover, USGS 3DEP elevation/slope/aspect/northness, and TerraClimate post-fire anomalies.
5. **Multi-Year Recovery Time Series:** Extract 5-year post-fire annual peak summer NDVI (2020–2024).
6. **Severity-Stratified Sampling & OLS Regression:** Sample each severity class at a 200m nominal scale and model recovery rate as a function of burn severity, topography, pre-fire vegetation, and climate.

**Validation status:** Exploratory spectral-recovery prototype. Explicit UTM grid, MTBS binary reference comparison and buffered spatial prediction checks are included below. NDVI greening is not evidence of restored forest structure. Ordinary OLS uncertainty remains unvalidated; see residual diagnostics. Outputs should be regenerated after pipeline changes.


In [1]:
import os
import sys
import math
import ee
import geemap
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm

# Add repository root to python path
sys.path.append(os.path.abspath(".."))
from src.auth import init_ee
from src.config import (
    GEE_PROJECT, ASSETS, SEVERITY_BINS, SEVERITY_CLASSES,
    SEVERITY_PALETTE, DNBR_RAMP, DYNAMIC_WORLD_CLASSES, DYNAMIC_WORLD_PALETTE
)
from src.imagery import get_peak_summer_hls
from src.severity import compute_burn_severity
from src.terrain import get_terrain_covariates
from src.landcover import get_prefire_landcover
from src.timeseries import get_multiyear_recovery_stack, get_terraclimate_anomalies
from src.modeling import sample_fire_scar, calculate_recovery_slopes, fit_recovery_ols, compute_morans_i, buffered_spatial_cv

# Initialize Earth Engine with configured project
init_ee(GEE_PROJECT)

[auth] Earth Engine initialized successfully with project: epscor-ioc


## 1. Fire Boundary Ingestion
We query the **Shovel Creek Fire (2019)** perimeter from the accessible Alaska Fire History collection (`projects/ee-ssahoo2/assets/AK_fire_history`).

In [2]:
ak_fires = ee.FeatureCollection(ASSETS["ak_fire_history"])
matches = ak_fires.filter(ee.Filter.eq("FIREYEAR", "2019")).filter(ee.Filter.eq("NAME", "Shovel Creek"))
assert matches.size().getInfo() == 1, "Expected one Shovel Creek perimeter"
shovel_fire = matches.first()
fire_info = shovel_fire.getInfo() or {}
fire_props = fire_info.get("properties", {})
geom = shovel_fire.geometry()

# Shovel Creek lies in UTM zone 6N. This is a site-specific analysis grid.
analysis_crs = "EPSG:32606"
analysis_projection = ee.Projection(analysis_crs).atScale(30)
print("Analysis grid:", analysis_projection.getInfo())

print(f"Fire Name: {fire_props.get('NAME')}")
print(f"Fire Year: {fire_props.get('FIREYEAR')}")
print(f"Recorded Acres: {fire_props.get('ACRES', 0):,.2f}")
centroid_info = geom.centroid().getInfo() or {}
print(f"Centroid: {centroid_info.get('coordinates', [])}")

Analysis grid: {'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, 30, 0]}
Fire Name: Shovel Creek
Fire Year: 2019
Recorded Acres: 22,487.47
Centroid: [-148.3160772814723, 64.98500541494958]


## 2. NASA Harmonized Landsat-Sentinel (HLS) Compositing
We use NASA HLS (L30 + S30) to reduce sensor bandpass differences:
- **Pre-fire Baseline:** Peak summer June 15 inclusive – August 15 exclusive, 2018 (Year Y-1)
- **Extended Post-fire:** Peak summer June 15 inclusive – August 15 exclusive, 2020 (Year Y+1)

In [3]:
pre_fire_hls = get_peak_summer_hls(geom, 2018).setDefaultProjection(analysis_projection)
post_fire_hls = get_peak_summer_hls(geom, 2020).setDefaultProjection(analysis_projection)

print("Pre-fire 2018 Bands:", pre_fire_hls.bandNames().getInfo())
print("Post-fire 2020 Bands:", post_fire_hls.bandNames().getInfo())

Pre-fire 2018 Bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'NBR', 'NDVI']
Post-fire 2020 Bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'NBR', 'NDVI']


## 3. Burn Severity: dNBR & RdNBR Computation
Computing extended differenced NBR and relativized dNBR, binned into initial dNBR severity classes.

Class names are provisional bins, not evidence of ecological damage or regrowth. RdNBR uses a numerical safeguard near zero pre-fire NBR and is not a predictor in the current models. The fixed bins apply only to dNBR.

In [4]:
severity_img = compute_burn_severity(pre_fire_hls, post_fire_hls)
dnbr = severity_img.select("dNBR")
rdnbr = severity_img.select("RdNBR")
severity_class = severity_img.select("severity_class")

# Calculate zonal summary across the fire perimeter
stats = dnbr.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=geom,
    scale=30,
    crs=analysis_projection,
    maxPixels=1e8
).getInfo() or {}

print("Shovel Creek Extended dNBR Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v:.2f}")

Shovel Creek Extended dNBR Statistics:
  dNBR_max: 1064.40
  dNBR_mean: 437.34
  dNBR_min: -321.23


## 4. Interactive Map Visualization
Explore pre-fire, post-fire false color composites, continuous dNBR, and provisional dNBR classes requiring local calibration.

In [5]:
m = geemap.Map(center=[65.05, -148.42], zoom=11)
m.add_basemap("HYBRID")

# False Color (SWIR2, NIR, Red)
fc_vis = {"bands": ["SWIR2", "NIR", "Red"], "min": 0.0, "max": 0.4}
m.addLayer(pre_fire_hls, fc_vis, "Pre-Fire 2018 (False Color)")
m.addLayer(post_fire_hls, fc_vis, "Post-Fire 2020 (False Color)")

# Continuous dNBR Layer
dnbr_vis = {"min": -100, "max": 800, "palette": DNBR_RAMP}
m.addLayer(dnbr, dnbr_vis, "Continuous dNBR")

# Discrete Initial dNBR Severity Classification
class_vis = {"min": 0, "max": 5, "palette": SEVERITY_PALETTE}
m.addLayer(severity_class, class_vis, "Initial dNBR Severity Classification")

# Boundary Outline
outline = ee.Image().paint(ee.FeatureCollection([shovel_fire]), 0, 2)
m.addLayer(outline, {"palette": ["#FFFFFF"]}, "Fire Boundary Outline")

m

## 5. Environmental Covariates Extraction
- **USGS 3DEP Topography:** Elevation, Slope, Aspect, and Northness
- **Google Dynamic World 10m:** Pre-fire dominant landcover
- **TerraClimate:** Post-fire summer maximum temperature anomaly and precipitation anomaly

In [6]:
terrain = get_terrain_covariates(geom)
prefire_dw = get_prefire_landcover(geom, 2018)
climate_anom = get_terraclimate_anomalies(geom, 2020, 2024)

print("Terrain Bands:", terrain.bandNames().getInfo())
print("Dynamic World Bands:", prefire_dw.bandNames().getInfo())
print("Climate Anomaly Bands:", climate_anom.bandNames().getInfo())

Terrain Bands: ['elevation', 'slope', 'aspect', 'northness']
Dynamic World Bands: ['prefire_landcover', 'trees', 'shrub_and_scrub', 'grass']
Climate Anomaly Bands: ['temp_anomaly', 'precip_anomaly']


## 6. Multi-Year Vegetation Recovery Time Series (2020–2024)
We assemble annual NDVI from summer median-reflectance composites from Year Y+1 (2020) through 2024 to track recovery trajectories.

In [7]:
recovery_years = [2020, 2021, 2022, 2023, 2024]
ndvi_stack = get_multiyear_recovery_stack(geom, 2020, 2024)

# Build unified multi-band analysis stack
full_stack = ee.Image([
    dnbr, rdnbr, severity_class,
    terrain,
    prefire_dw,
    climate_anom,
    ndvi_stack
])

print("Full Feature Stack Bands:", full_stack.bandNames().getInfo())

Full Feature Stack Bands: ['dNBR', 'RdNBR', 'severity_class', 'elevation', 'slope', 'aspect', 'northness', 'prefire_landcover', 'trees', 'shrub_and_scrub', 'grass', 'temp_anomaly', 'precip_anomaly', 'NDVI_2020', 'NDVI_2021', 'NDVI_2022', 'NDVI_2023', 'NDVI_2024']


## 7. Severity-Stratified Sampling & OLS Regression
We draw a reproducible random sample within each severity class at a 200m nominal scale, then test the model residuals for remaining spatial autocorrelation.
Sampling uses EPSG:32606 at 200 m nominal scale with nearest-neighbor evaluation, not 200 m area averages or guaranteed point spacing. Class-balanced samples do not estimate area-weighted fire means.
We then fit exploratory multiple linear regression: `recovery_slope ~ dNBR + elevation + slope + northness + trees + temp_anomaly + precip_anomaly`.

In [8]:
print("Sampling spatial points across Shovel Creek scar...")
df_samples = sample_fire_scar(full_stack, geom, scale=200, max_points=2000, projection=analysis_projection)
print(f"Total valid sampled points: {len(df_samples):,}")

# Calculate linear recovery slope (delta NDVI / year) per point
df_samples["recovery_slope"] = calculate_recovery_slopes(df_samples, recovery_years)

# Mean recovery slope grouped by provisional dNBR class
severity_summary = df_samples.groupby("severity_class")["recovery_slope"].agg(["count", "mean", "std"]).reset_index()
severity_summary["class_name"] = severity_summary["severity_class"].map(lambda x: SEVERITY_CLASSES[int(x)] if int(x) < len(SEVERITY_CLASSES) else "Unknown")
print("\nRecovery Rate (delta NDVI/yr) by Severity Class:")
print(severity_summary[["severity_class", "class_name", "count", "mean", "std"]])

Sampling spatial points across Shovel Creek scar...
Total valid sampled points: 1,573

Recovery Rate (delta NDVI/yr) by Severity Class:
   severity_class     class_name  count      mean       std
0               1       Unburned    278  0.000371  0.004827
1               2   Low Severity    296  0.006777  0.007278
2               3   Moderate-Low    333  0.012637  0.015297
3               4  Moderate-High    333  0.022841  0.014570
4               5  High Severity    333  0.032904  0.013523


In [9]:
# Fit OLS Regression Model
predictors = ["dNBR", "elevation", "slope", "northness", "trees", "temp_anomaly", "precip_anomaly"]
ols_model, vif_table = fit_recovery_ols(df_samples, predictors, target_var="recovery_slope")

print("Exploratory fit: ordinary OLS p-values/intervals are not spatially validated.")
print(ols_model.summary())
print("\n--- Multicollinearity Diagnostic: Variance Inflation Factors (VIF) ---")
print(vif_table)

Exploratory fit: ordinary OLS p-values/intervals are not spatially validated.
                            OLS Regression Results                            
Dep. Variable:         recovery_slope   R-squared:                       0.548
Model:                            OLS   Adj. R-squared:                  0.546
Method:                 Least Squares   F-statistic:                     270.8
Date:                Mon, 21 Sep 2026   Prob (F-statistic):          2.48e-264
Time:                        14:37:47   Log-Likelihood:                 4829.7
No. Observations:                1573   AIC:                            -9643.
Df Residuals:                    1565   BIC:                            -9601.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------

In [10]:
# Spatial Autocorrelation Diagnostic (Moran's I on Residuals)
model_rows = ols_model.model.data.row_labels
coords = np.asarray(df_samples.loc[model_rows, ["longitude", "latitude"]].values)
residuals = np.asarray(ols_model.resid.values)
moran_results = compute_morans_i(residuals, coords, k_neighbors=8, permutations=999)

print("Spatial Autocorrelation Diagnostic on Regression Residuals:")
for k, v in moran_results.items():
    print(f"  {k}: {v}")

Spatial Autocorrelation Diagnostic on Regression Residuals:
  morans_i: 0.3898
  expected_i: -0.0006
  p_value: 0.001
  permutations: 999


## 8. Interactive Plotly Visualizations
Plotting the mean NDVI recovery trajectory across severity classes and the regression coefficients.

In [11]:
# 1. Multi-Year Recovery Trajectory by Severity Class
ndvi_cols = [f"NDVI_{yr}" for yr in recovery_years]
traj_df = df_samples.groupby("severity_class")[ndvi_cols].mean().reset_index()
traj_long = traj_df.melt(id_vars="severity_class", var_name="Year", value_name="Mean_NDVI")
traj_long["Year"] = traj_long["Year"].str.replace("NDVI_", "").astype(int)
traj_long["Severity"] = traj_long["severity_class"].map(lambda x: SEVERITY_CLASSES[int(x)] if int(x) < len(SEVERITY_CLASSES) else "Unknown")

fig_traj = px.line(
    traj_long, x="Year", y="Mean_NDVI", color="Severity", markers=True,
    title="Shovel Creek Fire (2019): Post-Fire Summer NDVI Trajectory (2020-2024)",
    labels={"Mean_NDVI": "Mean Summer NDVI", "Year": "Growing Season"},
    template="plotly_white"
)
fig_traj.update_layout(hovermode="x unified", font=dict(family="Arial", size=12))
fig_traj.show()

Figure exported locally: figure_1.html (browser rendering unverified)


In [12]:
# Exploratory coefficients only: residual dependence invalidates naive intervals.
coef_df = pd.DataFrame({
    "Predictor": ols_model.params.index[1:],
    "Coefficient": ols_model.params.values[1:]
}).sort_values(by="Coefficient")
fig_coef = px.scatter(coef_df, x="Coefficient", y="Predictor",
    title="Exploratory OLS coefficients; predictor units differ; uncertainty unvalidated",
    template="plotly_white")
fig_coef.add_vline(x=0, line_dash="dash", line_color="gray")
fig_coef.show()


Figure exported locally: figure_2.html (browser rendering unverified)


## 9. MTBS reference-product comparison
Use the [Alaska 2019 MTBS severity mosaic](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_MTBS_annual_burn_severity_mosaics_v1), restricted to the intersection of the matched MTBS event and Alaska Fire History boundaries. Exclude background (0) and non-mapping areas (6); evaluate both images on the same 30 m UTM grid using nearest-neighbor class resampling.

Compare **high versus other mapped severity**, not our six class names against MTBS's five. Test our provisional threshold 660, the event's threshold 700, and 714 (=700 + the event's 14-unit background offset). Applying that offset is a sensitivity scenario, not a calibrated correction for our seasonal composites. Acquisition dates remain different (MTBS 2018-07-22 / 2020-06-09); agreement is neither field accuracy nor independent validation. Annual mosaics may combine overlapping fires, so check event overlap.


In [13]:
mtbs_events = ee.FeatureCollection(ASSETS["mtbs"])
mtbs_match = mtbs_events.filter(ee.Filter.eq("Event_ID", "AK6498714841520190621"))
assert mtbs_match.size().getInfo() == 1, "Expected one MTBS event"
mtbs_event = ee.Feature(mtbs_match.first())
print("MTBS event metadata:", mtbs_event.toDictionary().getInfo())
comparison_geom = geom.intersection(mtbs_event.geometry(), 1)
reference = ee.Image("USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1/mtbs_AK_2019").select("Severity")
mapped = reference.gte(1).And(reference.lte(5))
paired = mapped.And(dnbr.mask())
area = ee.Image.pixelArea()
comparison_kwargs = dict(geometry=comparison_geom, crs=analysis_projection, scale=30, maxPixels=1e8)
comparison_areas = ee.Image([
    area.rename("intersection_m2"),
    area.updateMask(mapped).rename("reference_mapped_m2"),
    area.updateMask(paired).rename("paired_m2"),
]).reduceRegion(ee.Reducer.sum(), **comparison_kwargs).getInfo()
print("Comparison support:", comparison_areas)
def high_severity_agreement(index, support, threshold):
    # Code: 0 neither high; 1 HLS only; 2 MTBS only; 3 both high.
    pair_code = reference.eq(4).multiply(2).add(index.gt(threshold)).rename("pair")
    grouped = area.rename("area").addBands(pair_code).updateMask(support).reduceRegion(
        ee.Reducer.sum().group(1, "pair"), **comparison_kwargs).getInfo()["groups"]
    areas = {int(v["pair"]): v["sum"] for v in grouped}
    neither, hls_only, mtbs_only, both = [areas.get(i, 0) for i in range(4)]
    total = sum(areas.values())
    assert total > 0, "No paired valid pixels"
    return dict(threshold=threshold, neither_m2=neither,
        hls_only_m2=hls_only, mtbs_only_m2=mtbs_only, both_high_m2=both,
        agreement=(neither + both) / total,
        high_iou=both / (both + hls_only + mtbs_only) if both + hls_only + mtbs_only else np.nan, paired_m2=total)
mtbs_comparison = [high_severity_agreement(dnbr, paired, threshold) for threshold in [660, 700, 714]]
assert all(np.isclose(row["paired_m2"], comparison_areas["paired_m2"]) for row in mtbs_comparison)
print(pd.DataFrame(mtbs_comparison).to_string(index=False))
m.addLayer(reference.updateMask(mapped).clip(comparison_geom),
    {"min": 1, "max": 5, "palette": ["006400", "7fffd4", "ffff00", "ff0000", "7fff00"]},
    "MTBS 2019 reference (different acquisition dates)")


MTBS event metadata: {'Asmnt_Type': 'Extended', 'BurnBndAc': 22128, 'BurnBndLat': '64.988', 'BurnBndLon': '-148.3', 'Comment': '', 'Event_ID': 'AK6498714841520190621', 'High_T': 700, 'Ig_Date': 1561100400000, 'IncGreen_T': -150, 'Incid_Name': 'SHOVEL CREEK', 'Incid_Type': 'Wildfire', 'Low_T': 80, 'Map_ID': 10016695, 'Map_Prog': 'MTBS', 'Mod_T': 378, 'NoData_T': -970, 'Perim_ID': '', 'Post_ID': 'B06WVT20200609_30m', 'Pre_ID': 'A06WVT20180722_30m', 'dNBR_offst': 14, 'dNBR_stdDv': 18, 'irwinID': '9155A802-F466-4D32-AFA3-F0B2D823B2EB'}
Comparison support: {'intersection_m2': 87070102.9007011, 'paired_m2': 86168840.173565, 'reference_mapped_m2': 86169740.82914361}
 threshold   neither_m2  hls_only_m2  mtbs_only_m2  both_high_m2  agreement  high_iou    paired_m2
       660 5.002297e+07 3.423795e+06  1.594688e+07  1.677519e+07   0.775201  0.464097 8.616884e+07
       700 5.161773e+07 1.829038e+06  1.971694e+07  1.300514e+07   0.749956  0.376403 8.616884e+07
       714 5.201941e+07 1.427356e+0

## 10. Buffered spatial prediction and response sensitivity
Leave one occupied square block out at a time; remove training observations closer than the specified buffer to any test observation. Use the same predictor list without tuning. The baseline predicts the **training fold's mean**. Test 2/4 km blocks and 0/1/2 km buffers as exploratory sensitivity choices, not estimated independence distances. Metrics weight the sampled points equally, not fire area. These assess within-fire prediction; held-out fires are still needed for transfer claims.

Repeat using the 2021–2024 slope to remove 2020 from the response, because dNBR also uses 2020 imagery. This tests shared-year sensitivity; it does not remove all confounding or establish causation. Report every design, not only the best score.


In [14]:
validation_scores = []
spatial_predictions = {}
spatial_folds = {}
for start in [2020, 2021]:
    validation_data = df_samples.copy()
    validation_data["recovery_slope"] = calculate_recovery_slopes(validation_data,
        [year for year in recovery_years if year >= start])
    for block_m in [2000, 4000]:
        for buffer_m in [0, 1000, 2000]:
            predictions, folds = buffered_spatial_cv(validation_data, predictors, block_m, buffer_m)
            key = f"{start}_block{block_m}_buffer{buffer_m}"
            spatial_predictions[key], spatial_folds[key] = predictions, folds
            errors = predictions.observed - predictions.predicted
            baseline_errors = predictions.observed - predictions.baseline
            tss = ((predictions.observed - predictions.observed.mean()) ** 2).sum()
            validation_scores.append(dict(start_year=start, block_m=block_m, buffer_m=buffer_m,
                blocks=len(folds), n=len(predictions), rmse=np.sqrt(np.mean(errors ** 2)),
                mae=np.mean(np.abs(errors)), baseline_rmse=np.sqrt(np.mean(baseline_errors ** 2)),
                pooled_r2=1 - (errors ** 2).sum() / tss,
                skill_vs_training_mean=1 - (errors ** 2).sum() / (baseline_errors ** 2).sum(),
                min_train_test_m=folds.min_train_test_m.min()))
validation_scores = pd.DataFrame(validation_scores)
print(validation_scores.to_string(index=False))


 start_year  block_m  buffer_m  blocks    n     rmse      mae  baseline_rmse  pooled_r2  skill_vs_training_mean  min_train_test_m
       2020     2000         0      48 1573 0.011481 0.008308       0.016819   0.527190                0.534068             200.0
       2020     2000      1000      48 1573 0.011543 0.008372       0.016933   0.522067                0.535322            1000.0
       2020     2000      2000      48 1573 0.011675 0.008468       0.017094   0.511026                0.533496            2000.0
       2020     4000         0      17 1573 0.011637 0.008453       0.016959   0.514238                0.529160             200.0
       2020     4000      1000      17 1573 0.011632 0.008484       0.017046   0.514652                0.534375            1000.0
       2020     4000      2000      17 1573 0.011800 0.008542       0.017206   0.500504                0.529651            2000.0
       2021     2000         0      48 1573 0.010429 0.007822       0.011715   0.198722   

## 11. Acquisition-date sensitivity on identical pixels
Recompute HLS dNBR using only the MTBS calendar dates: 2018-07-22 and 2020-06-09. Compare seasonal and date-matched HLS on the **same common valid mask**, so changing coverage does not drive the comparison. HLS harmonization, screening and MTBS processing still differ; matching dates does not reproduce the official continuous dNBR product. Threshold/offset scenarios remain sensitivity checks, not fitted calibration.


In [15]:
date_pre = get_peak_summer_hls(geom, 2018, "2018-07-22", "2018-07-23")
date_post = get_peak_summer_hls(geom, 2020, "2020-06-09", "2020-06-10")
date_dnbr = compute_burn_severity(date_pre, date_post).select("dNBR")
date_support = paired.And(date_dnbr.mask())
date_comparison = []
for method, index in [("summer_composite", dnbr), ("MTBS_calendar_dates", date_dnbr)]:
    for threshold in [660, 700, 714]:
        row = high_severity_agreement(index, date_support, threshold)
        row["method"] = method
        date_comparison.append(row)
assert all(np.isclose(row["paired_m2"], date_comparison[0]["paired_m2"]) for row in date_comparison)
print("Common valid area (m²):", date_comparison[0]["paired_m2"])
print(pd.DataFrame(date_comparison)[["method", "threshold", "agreement", "high_iou"]].to_string(index=False))


Common valid area (m²): 79906486.20077975
             method  threshold  agreement  high_iou
   summer_composite        660   0.772684  0.475910
   summer_composite        700   0.744757  0.385757
   summer_composite        714   0.733035  0.350091
MTBS_calendar_dates        660   0.823464  0.682146
MTBS_calendar_dates        700   0.862176  0.728626
MTBS_calendar_dates        714   0.875540  0.745599


## 12. Which predictors improve held-out predictions?
Prespecified comparisons: dNBR alone; dNBR + terrain + vegetation without climate; the existing full model; the full model without dNBR; initial 2020 NDVI alone; and the full model plus initial NDVI. Use identical rows and spatial folds for every model, retaining all 2/4 km block and 0/1/2 km buffer scenarios for both response windows.

These are **exploratory predictive contributions**, not causal feature importance or statistical significance tests. Initial NDVI shares 2020 with the 2020–2024 response; the later-start response avoids this direct shared-year input but is a different estimand. We have already inspected this fire, so favorable scores must be checked on other fires before model selection. The 0 km buffer is a diagnostic, not evidence of independence.


In [16]:
predictor_sets = {
    "dNBR_only": ["dNBR"],
    "without_climate": ["dNBR", "elevation", "slope", "northness", "trees"],
    "full": predictors,
    "without_dNBR": [p for p in predictors if p != "dNBR"],
    "initial_NDVI_only": ["NDVI_2020"],
    "full_plus_initial_NDVI": [*predictors, "NDVI_2020"],
}
ablation_rows = []
ablation_predictions = {}
ablation_folds = {}
for start in [2020, 2021]:
    data = df_samples.copy()
    data["recovery_slope"] = calculate_recovery_slopes(data,
        [year for year in recovery_years if year >= start])
    for block_m in [2000, 4000]:
        for buffer_m in [0, 1000, 2000]:
            reference_folds = None
            for name, variables in predictor_sets.items():
                held_out, folds = buffered_spatial_cv(data, variables, block_m, buffer_m)
                if reference_folds is not None:
                    pd.testing.assert_frame_equal(folds, reference_folds)
                reference_folds = folds
                key = f"{start}_{block_m}_{buffer_m}_{name}"
                ablation_predictions[key], ablation_folds[key] = held_out, folds
                errors = held_out.observed - held_out.predicted
                baseline_errors = held_out.observed - held_out.baseline
                tss = ((held_out.observed - held_out.observed.mean()) ** 2).sum()
                ablation_rows.append(dict(start_year=start, block_m=block_m, buffer_m=buffer_m,
                    model=name, n=len(held_out), blocks=len(folds),
                    rmse=np.sqrt(np.mean(errors ** 2)), mae=np.mean(np.abs(errors)),
                    baseline_rmse=np.sqrt(np.mean(baseline_errors ** 2)),
                    pooled_r2=1 - (errors ** 2).sum() / tss))
ablation_scores = pd.DataFrame(ablation_rows)
print("All models evaluated on the same rows and folds; sample-weighted, not area-weighted.")
print(ablation_scores.groupby(["start_year", "model"])[["pooled_r2", "rmse"]]
    .agg(["min", "max"]).to_string())


All models evaluated on the same rows and folds; sample-weighted, not area-weighted.
                                  pooled_r2                rmse          
                                        min       max       min       max
start_year model                                                         
2020       dNBR_only               0.439361  0.473771  0.012112  0.012502
           full                    0.500504  0.527190  0.011481  0.011800
           full_plus_initial_NDVI  0.590644  0.633255  0.010111  0.010683
           initial_NDVI_only       0.576697  0.595126  0.010624  0.010863
           without_climate         0.437404  0.499861  0.011808  0.012523
           without_dNBR           -0.035713  0.057653  0.016208  0.016992
2021       dNBR_only               0.084604  0.108082  0.011003  0.011147
           full                    0.179959  0.201439  0.010412  0.010551
           full_plus_initial_NDVI  0.217927  0.239286  0.010162  0.010304
           initial_NDVI_onl

In [17]:
fig_ablation = px.scatter(ablation_scores, x="model", y="pooled_r2", color="start_year",
    symbol="block_m", hover_data=["buffer_m", "rmse", "baseline_rmse"],
    title="Shovel Creek predictor checks: all spatial designs retained",
    labels={"model": "Predictor set", "pooled_r2": "Held-out pooled R²", "start_year": "Response starts"},
    template="plotly_white")
fig_ablation.add_hline(y=0, line_dash="dash", line_color="gray")
fig_ablation.show()
